
# Preprocesamiento del dataset base depurado para red neuronal

Este notebook reproduce el pipeline utilizado para preparar los archivos de entrenamiento, validación y testeo.

**Target:** `PERMANENCIA_NORMALIZADA`
- 0 = NO DESERCIÓN
- 1 = DESERCIÓN

**Split estratificado:** 70% train, 15% validation, 15% test.

Todas las transformaciones se ajustan únicamente con train para evitar *data leakage*.


In [ ]:

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin

RANDOM_STATE = 42
TARGET = "PERMANENCIA_NORMALIZADA"

# CAMBIA ESTA RUTA
DATASET_PATH = "data/Datos_depurados_base.xlsx"

OUTPUT_DIR = Path("outputs/preprocesamiento_salida")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

dataset_path = Path(DATASET_PATH)

if dataset_path.suffix.lower() in [".xlsx", ".xls"]:
    df = pd.read_excel(dataset_path)
elif dataset_path.suffix.lower() == ".csv":
    df = pd.read_csv(dataset_path, encoding="utf-8-sig")
else:
    raise ValueError("Formato no soportado. Use CSV/XLSX/XLS.")

print("Dimensiones:", df.shape)
display(df.head())

if TARGET not in df.columns:
    raise KeyError(f"No se encontró {TARGET}")


In [ ]:

COUNT_COLS = [
    "Cursos matriculados",
    "Cursos aprobados",
    "Cursos retirados",
    "Cursos reprobados",
    "Cursos desertados",
    "Créditos matriculados sin retiro",
    "Créditos aprobados",
    "Créditos retirados",
    "Créditos reprobados",
    "Créditos desertados",
    "Cursos repetidos",
    "Cantidad de carreras"
]

PERCENT_COLS = [
    "% cursos aprobados",
    "% cursos reprobados",
    "% cursos desertados",
    "% créditos aprobados",
    "% créditos reprobados",
    "% créditos desertados"
]

FORCE_NUMERIC = ["Edad"]
FORCE_CATEGORICAL = ["Código Índice de Urbanidad"]

count_cols_present = [c for c in COUNT_COLS if c in df.columns]
percent_cols_present = [c for c in PERCENT_COLS if c in df.columns]


In [ ]:

df_work = df.copy()
df_work["y"] = pd.to_numeric(df_work[TARGET], errors="coerce").astype("Int64")

if df_work["y"].isna().any():
    raise ValueError("El target contiene valores no convertibles a 0/1.")

vals = sorted(df_work["y"].unique().tolist())
if vals != [0, 1]:
    raise ValueError(f"Se esperaban [0,1], se encontró: {vals}")

df_work["y"] = df_work["y"].astype(int)

print(df_work["y"].value_counts())
print((df_work["y"].value_counts(normalize=True)*100).round(4))


In [ ]:

y_all = df_work["y"].values

train_idx, temp_idx = train_test_split(
    df_work.index,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_all
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df_work.loc[temp_idx, "y"]
)

print("Train:", len(train_idx))
print("Val  :", len(val_idx))
print("Test :", len(test_idx))


In [ ]:

def normalize_object_series(series):
    return series.apply(
        lambda x: np.nan if pd.isna(x)
        else re.sub(r"\s+", " ", str(x).strip())
    )

df_clean = df_work.copy()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = normalize_object_series(df_clean[col])

for col in df_clean.columns:
    if isinstance(col, str) and "%" in col:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean["split"] = "train"
df_clean.loc[val_idx, "split"] = "val"
df_clean.loc[test_idx, "split"] = "test"


In [ ]:

train_df = df_clean[df_clean["split"]=="train"].copy()
val_df   = df_clean[df_clean["split"]=="val"].copy()
test_df  = df_clean[df_clean["split"]=="test"].copy()

X_train_raw = train_df.drop(columns=["y","split"])
y_train = train_df["y"].astype(int).values

X_val_raw = val_df.drop(columns=["y","split"])
y_val = val_df["y"].astype(int).values

X_test_raw = test_df.drop(columns=["y","split"])
y_test = test_df["y"].astype(int).values

for X in (X_train_raw, X_val_raw, X_test_raw):
    if TARGET in X.columns:
        X.drop(columns=[TARGET], inplace=True)


In [ ]:

force_numeric = [c for c in FORCE_NUMERIC if c in X_train_raw.columns]
force_categorical = [c for c in FORCE_CATEGORICAL if c in X_train_raw.columns]

object_cols = X_train_raw.select_dtypes(include=["object"]).columns.tolist()
numeric_candidates = X_train_raw.columns.difference(object_cols).tolist()

low_card_numeric_as_cat = []

for col in numeric_candidates:
    if col in force_numeric or col in count_cols_present or col in percent_cols_present:
        continue

    nunique = X_train_raw[col].nunique(dropna=True)

    if nunique <= 30 and nunique / len(X_train_raw) < 0.10:
        low_card_numeric_as_cat.append(col)

categorical_cols = sorted(set(
    object_cols + low_card_numeric_as_cat + force_categorical
))
categorical_cols = [c for c in categorical_cols if c not in force_numeric]

count_cols_use = [
    c for c in count_cols_present
    if c in X_train_raw.columns and c not in categorical_cols
]

percent_cols_use = [
    c for c in percent_cols_present
    if c in X_train_raw.columns and c not in categorical_cols
]

other_numeric_cols = [
    c for c in X_train_raw.columns
    if c not in categorical_cols
    and c not in count_cols_use
    and c not in percent_cols_use
]

print("Cantidades:", count_cols_use)
print("Porcentajes:", percent_cols_use)
print("Numéricas:", other_numeric_cols)
print("Categóricas:", categorical_cols)


In [ ]:

class ClipTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lower_q=0.0, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.upper_ = np.nanquantile(X, self.upper_q, axis=0)
        self.lower_ = (
            np.nanquantile(X, self.lower_q, axis=0)
            if self.lower_q > 0 else None
        )
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        if self.lower_ is not None:
            X = np.maximum(X, self.lower_)
        return np.minimum(X, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if input_features is not None else [])


In [ ]:

counts_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clip", ClipTransformer(upper_q=0.99)),
    ("log1p", FunctionTransformer(
        lambda x: np.log1p(np.maximum(x, 0)),
        feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler())
])

percent_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("to_proportion", FunctionTransformer(
        lambda x: x/100.0,
        feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler())
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("astype_str", FunctionTransformer(
        lambda x: x.astype(str),
        feature_names_out="one-to-one"
    )),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


In [ ]:

preprocess = ColumnTransformer(
    transformers=[
        ("counts", counts_transformer, count_cols_use),
        ("percent", percent_transformer, percent_cols_use),
        ("numeric", numeric_transformer, other_numeric_cols),
        ("categorical", categorical_transformer, categorical_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# FIT SOLO EN TRAIN
preprocess.fit(X_train_raw)

X_train = preprocess.transform(X_train_raw).astype(np.float32)
X_val   = preprocess.transform(X_val_raw).astype(np.float32)
X_test  = preprocess.transform(X_test_raw).astype(np.float32)

feature_names = preprocess.get_feature_names_out()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


In [ ]:

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]
assert not np.isnan(X_train).any()
assert not np.isnan(X_val).any()
assert not np.isnan(X_test).any()
assert not np.isinf(X_train).any()
assert not np.isinf(X_val).any()
assert not np.isinf(X_test).any()

print("Validaciones OK")


In [ ]:

train_ready = pd.DataFrame(X_train, columns=feature_names)
val_ready   = pd.DataFrame(X_val, columns=feature_names)
test_ready  = pd.DataFrame(X_test, columns=feature_names)

train_ready.insert(0, "y", y_train)
val_ready.insert(0, "y", y_val)
test_ready.insert(0, "y", y_test)

display(train_ready.head())


In [ ]:

train_path = OUTPUT_DIR / "base_X_train_ready.csv"
val_path   = OUTPUT_DIR / "base_X_val_ready.csv"
test_path  = OUTPUT_DIR / "base_X_test_ready.csv"

train_ready.to_csv(train_path, index=False, encoding="utf-8-sig")
val_ready.to_csv(val_path, index=False, encoding="utf-8-sig")
test_ready.to_csv(test_path, index=False, encoding="utf-8-sig")

print("Generados:")
print(train_path)
print(val_path)
print(test_path)


In [ ]:

metadata = {
    "source_file": str(DATASET_PATH),
    "target_col": TARGET,
    "target_mapping": {
        "0": "NO DESERCIÓN",
        "1": "DESERCIÓN"
    },
    "random_state": RANDOM_STATE,
    "splits": {
        "train": int(len(train_idx)),
        "validation": int(len(val_idx)),
        "test": int(len(test_idx))
    },
    "n_raw_features_used": int(X_train_raw.shape[1]),
    "n_features_after_preprocess": int(len(feature_names)),
    "count_cols_log1p": count_cols_use,
    "percent_cols_div100": percent_cols_use,
    "numeric_cols_scaled": other_numeric_cols,
    "categorical_cols_onehot": categorical_cols,
    "forced_numeric_cols": force_numeric,
    "forced_categorical_cols": force_categorical
}

metadata_path = OUTPUT_DIR / "preprocess_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(metadata_path)
